In [46]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [47]:
import os, argparse
from mne_bids import BIDSPath, read_raw_bids
from TUHProcessingJob import _process_subject
import pandas as pd
import dask.dataframe as dd

In [48]:
bids_path = BIDSPath(root="~/mycontainer/TUH_BIDS_Converted", subject="aaaaahhh", session="s0012008", processing="02tcple", recording="t000", task="rest", datatype="eeg", suffix="eeg", extension=".edf")
TUH_patient = read_raw_bids(bids_path, verbose=True)

print(TUH_patient.ch_names)
print(len(TUH_patient.ch_names))

subfolders = [f.path for f in os.scandir("/home/azureuser/mycontainer/TUH_BIDS_Converted") if f.is_dir() ]
subjects = [x.split("/")[-1][4:] for x in subfolders]
print(f"Found {len(subjects)} subjects")

Extracting EDF parameters from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_task-rest_proc-02tcple_rec-t000_eeg.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


Reading channel info from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_task-rest_proc-02tcple_rec-t000_channels.tsv.
Reading electrode coords from /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_space-CapTrak_electrodes.tsv.
Not fully anonymizing info - keeping his_id, sex, and hand info
['Fp1', 'Fp2', 'F3', 'F4', 'C3', 'C4', 'A1', 'A2', 'P3', 'P4', 'O1', 'O2', 'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'Fz', 'Cz', 'Pz']
21
Found 14987 subjects


In [49]:
parser = argparse.ArgumentParser(description='Preprocess TUH EEG data')
parser.add_argument('--input_directory', type=str, help='Path to the directory containing the TUH EEG data', default="/home/azureuser/mycontainer/TUH_BIDS_Converted")
parser.add_argument('--output_dir', type=str, help='Path to the directory where the preprocessed data will be stored', default="./")

args, unknownargs = parser.parse_known_args()

In [50]:
print(args.input_directory, args.output_dir)

/home/azureuser/mycontainer/TUH_BIDS_Converted ./


In [51]:
# Create an empty Pandas DataFrame with the specified columns
# Define the columns for the empty DataFrame
columns = ['bids_path', 'error_message'] 
empty_df = pd.DataFrame(columns=columns)
failed_files = dd.from_pandas(empty_df, npartitions=8)

In [52]:
subject, mean, std = _process_subject(args, "aaaaahhh", failed_files)

Processing subject: aaaaahhh
Processing /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0012008/eeg/sub-aaaaahhh_ses-s0012008_task-rest_proc-02tcple_rec-t000_eeg.edf
Not setting metadata
23 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 23 events and 15000 original time points ...
0 bad epochs dropped
Processing /home/azureuser/mycontainer/TUH_BIDS_Converted/sub-aaaaahhh/ses-s0022009/eeg/sub-aaaaahhh_ses-s0022009_task-rest_proc-02tcple_rec-t001_eeg.edf
Not setting metadata
21 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 21 events and 15000 original time points ...
0 bad epochs dropped


In [53]:
print(f"Subject: {subject}, Mean: {mean.shape}, Std: {std.shape}")

Subject: aaaaahhh, Mean: (19,), Std: (19,)
